In [ ]:
!pip install --upgrade pip
!pip install transformers datasets evaluate scikit-learn

In [ ]:
import transformers
import datasets
import evaluate
import sklearn

In [ ]:
from datasets import load_dataset, concatenate_datasets, DatasetDict

languages = ['de', 'en', 'es', 'fr', 'it', 'nl', 'pl', 'pt', 'ru', 'zh']
split_map = {'train': 'train', 'validation': 'dev', 'test': 'test'}
dataset = {}

for split_name, actual_split in split_map.items():
    split_datasets = [load_dataset("PhilipMay/stsb_multi_mt", lang, split=actual_split) for lang in languages]
    dataset[split_name] = concatenate_datasets(split_datasets)

dataset = DatasetDict(dataset)

In [ ]:
def normalize_scores(example):
    example["labels"] = float(example["similarity_score"]) / 5.0
    return example

# Apply to all splits
for split in dataset:
    dataset[split] = dataset[split].map(normalize_scores)

In [ ]:
from transformers import BertTokenizerFast

tokenizer = BertTokenizerFast.from_pretrained("bert-base-multilingual-cased")

def tokenize_function(example):
    return tokenizer(
        example["sentence1"],
        example["sentence2"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

# Tokenize all splits
for split in dataset:
    dataset[split] = dataset[split].map(tokenize_function, batched=True)

# Set the format to PyTorch tensors
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
# ✅ Metric Imports
import evaluate
pearson = evaluate.load("pearsonr")
spearman = evaluate.load("spearmanr")

# ✅ Metric Computation Function
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.squeeze()
    return {
        "pearson": pearson.compute(predictions=predictions, references=labels)["pearsonr"],
        "spearman": spearman.compute(predictions=predictions, references=labels)["spearmanr"]
    }

# ✅ Corrected import (you had a small typo: `rom` ➝ `from`)
from transformers import TrainingArguments, Trainer, AutoModelForSequenceClassification

# ✅ Training Arguments
training_args = TrainingArguments(
    output_dir="./results",                   # Where to save model checkpoints
    evaluation_strategy="epoch",              # Evaluate every epoch
    save_strategy="epoch",                    # Save every epoch
    learning_rate=2e-5,                       # Typical for BERT-like models
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="pearson",
    save_total_limit=2,
    logging_dir="./logs",
    logging_steps=100,
)

# ✅ Model Definition
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-multilingual-cased",
    num_labels=1  # Regression = single output neuron
)

# ✅ If your dataset is named `dataset`, not `tokenized_dataset`
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    compute_metrics=compute_metrics,
)

# ✅ Train the Model
trainer.train()

In [ ]:
trainer.save_model("mbert-stsb-multilingual")
tokenizer.save_pretrained("mbert-stsb-multilingual")

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained("mbert-stsb-multilingual")
tokenizer = AutoTokenizer.from_pretrained("mbert-stsb-multilingual")

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

In [ ]:
from datasets import load_dataset, concatenate_datasets
from tqdm.auto import tqdm

languages = ['de', 'en', 'es', 'fr', 'it', 'nl', 'pl', 'pt', 'ru', 'zh']
test_sets = [load_dataset("PhilipMay/stsb_multi_mt", lang, split="test") for lang in languages]
combined_test = concatenate_datasets(test_sets)

# Normalize and tokenize if needed
def normalize(example):
    example["labels"] = float(example["similarity_score"]) / 5.0
    return example

combined_test = combined_test.map(normalize)

def tokenize(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True, padding="max_length", max_length=128)

combined_test = combined_test.map(tokenize, batched=True)
combined_test.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
from torch.utils.data import DataLoader

dataloader = DataLoader(combined_test, batch_size=32)
predictions = []

with torch.no_grad():
    for batch in tqdm(dataloader, desc="Evaluating"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        logits = outputs.logits.squeeze().detach().cpu().numpy()
        predictions.extend(logits)

# De-normalize predicted scores
predicted_scores = [min(max(score, 0), 1) * 5.0 for score in predictions]  # Clamp & scale back to [0, 5]

In [ ]:
import numpy as np
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_squared_error

gold_scores = combined_test["labels"]
gold_scores = np.array(gold_scores) * 5.0  # De-normalize gold scores

preds = np.array(predicted_scores)

# Cosine Similarity
dot = np.dot(preds, gold_scores)
cosine_sim = dot / (np.linalg.norm(preds) * np.linalg.norm(gold_scores))

# All metrics
metrics = {
    "Pearson Correlation": pearsonr(preds, gold_scores)[0],
    "Spearman Correlation": spearmanr(preds, gold_scores)[0],
    "Mean Squared Error": mean_squared_error(gold_scores, preds),
    "Cosine Similarity": cosine_sim,
    "Average Correlation": (pearsonr(preds, gold_scores)[0] + spearmanr(preds, gold_scores)[0]) / 2
}

print("\nEvaluation Metrics:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "sentence1": combined_test["sentence1"],
    "sentence2": combined_test["sentence2"],
    "gold_similarity_score": gold_scores,
    "predicted_similarity_score": preds
})

df.to_csv("mbert_sts_evaluation.csv", index=False)
print("Saved evaluation results to 'mbert_sts_evaluation.csv'")

In [ ]:
import shutil
shutil.make_archive("mbert-stsb-multilingual", 'zip', "mbert-stsb-multilingual")

In [ ]:
from google.colab import files
files.download("mbert-stsb-multilingual.zip")

In [ ]:
files.download("mbert_sts_evaluation.csv")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Save model to Drive
!cp -r mbert-stsb-multilingual /content/drive/MyDrive/

# Save CSV to Drive
!cp mbert_sts_evaluation.csv /content/drive/MyDrive/

In [ ]:
from datasets import load_dataset, concatenate_datasets
from tqdm.auto import tqdm

languages = ['de', 'en', 'es', 'fr', 'it', 'nl', 'pl', 'pt', 'ru', 'zh']
dev_sets = [load_dataset("PhilipMay/stsb_multi_mt", lang, split="dev") for lang in languages]
combined_test1 = concatenate_datasets(dev_sets)

# Normalize and tokenize if needed
def normalize(example):
    example["labels"] = float(example["similarity_score"]) / 5.0
    return example

combined_test1 = combined_test1.map(normalize)

def tokenize(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True, padding="max_length", max_length=128)

combined_test1 = combined_test1.map(tokenize, batched=True)
combined_test1.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
from torch.utils.data import DataLoader

dataloader = DataLoader(combined_test1, batch_size=32)
predictions = []

with torch.no_grad():
    for batch in tqdm(dataloader, desc="Evaluating"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        logits = outputs.logits.squeeze().detach().cpu().numpy()
        predictions.extend(logits)

# De-normalize predicted scores
predicted_scores = [min(max(score, 0), 1) * 5.0 for score in predictions]  # Clamp & scale back to [0, 5]

In [ ]:
import numpy as np
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_squared_error

gold_scores = combined_test1["labels"]
gold_scores = np.array(gold_scores) * 5.0  # De-normalize gold scores

preds = np.array(predicted_scores)

# Cosine Similarity
dot = np.dot(preds, gold_scores)
cosine_sim = dot / (np.linalg.norm(preds) * np.linalg.norm(gold_scores))

# All metrics
metrics = {
    "Pearson Correlation": pearsonr(preds, gold_scores)[0],
    "Spearman Correlation": spearmanr(preds, gold_scores)[0],
    "Mean Squared Error": mean_squared_error(gold_scores, preds),
    "Cosine Similarity": cosine_sim,
    "Average Correlation": (pearsonr(preds, gold_scores)[0] + spearmanr(preds, gold_scores)[0]) / 2
}

print("\nEvaluation Metrics:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")